## Build email extractor

In [1]:
from pydantic import BaseModel, Field, EmailStr
from typing import Optional, Literal
from pydantic_ai import Agent
from constants import MODEL_LARGE
from dotenv import load_dotenv

load_dotenv()


class EmailExtractor(BaseModel):
    sender_name: Optional[str]
    sender_email: EmailStr
    issue_category: Literal["billing", "damaged product", "technical", "other"]
    urgency: Literal["low", "medium", "high"]
    summary: str = Field(description="Summarize the email in 3-4 sentences")


email_extractor_agent = Agent(
    MODEL_LARGE,
    system_prompt="""
You are customer support agent, your task is to
                              extract relevant information from an email
                              """,
    output_type=EmailExtractor,
)

TODO: load in the data and test the agent on one email

In [2]:
import pandas as pd 

df = pd.read_json('emails_cleaned.json')
df

,inputs,expectations
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'excepted_response': '{""sender_name"": ""Erik L..."
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'excepted_response': '{""sender_name"": ""Maja B..."
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'excepted_response': '{""sender_name"": ""Oscar ..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'excepted_response': '{""sender_name"": ""Linnea..."


In [3]:
sample_mail = df.iloc[2]['inputs']['email']
sample_mail

"From: Oscar Johansson <oscar.johansson@yahoo.se>\nSubject: Cannot access my account for 3 days - Urgent help needed\n\nHello Support Team,\n\nI am reaching out because I have been completely locked out of my account for the past three days and I am running out of ideas on how to fix this on my own. The problem started on Monday evening when I tried to log in as usual but kept receiving an 'Invalid credentials' error despite being absolutely certain that I was entering the correct password.\n\nI followed the instructions on your website to reset my password, but the problem is that the password reset email never arrives in my inbox. I have checked my spam and junk folders multiple times, and there is nothing there either. I have attempted the reset process at least six or seven times across different browsers and even from my phone, but the result is always the same - no email arrives.\n\nThis is causing me real problems because I have important documents and data stored in my account 

In [4]:
result = await email_extractor_agent.run(sample_mail)
result

AgentRunResult(output=EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_category='technical', urgency='high', summary="Oscar Johansson has been unable to access his account for three days due to persistent 'Invalid credentials' errors. He has attempted password reset multiple times but reset emails are not arriving in his inbox (including spam/junk). He needs urgent access to important work documents before a Friday deadline and has tried various troubleshooting steps without success."))

In [5]:
result.output

EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_category='technical', urgency='high', summary="Oscar Johansson has been unable to access his account for three days due to persistent 'Invalid credentials' errors. He has attempted password reset multiple times but reset emails are not arriving in his inbox (including spam/junk). He needs urgent access to important work documents before a Friday deadline and has tried various troubleshooting steps without success.")

In [6]:
isinstance(result.output, BaseModel)

True

In [7]:
result.output.summary

"Oscar Johansson has been unable to access his account for three days due to persistent 'Invalid credentials' errors. He has attempted password reset multiple times but reset emails are not arriving in his inbox (including spam/junk). He needs urgent access to important work documents before a Friday deadline and has tried various troubleshooting steps without success."

TODO: show ground truth and compare

## Load in prompts from mlflow

In [8]:
from mlflow.genai import load_prompt

class EmailExtractor(BaseModel):
    sender_name: Optional[str]
    sender_email: EmailStr
    issue_category: Literal["billing", "damaged product", "technical", "other"]
    urgency: Literal["low", "medium", "high"] = Field(
        description=load_prompt("email-urgency-description").format()
    )
    summary: str = Field(
        description=load_prompt("summary-description").format(num_sentences=4)
    )


email_extractor_agent = Agent(
    MODEL_LARGE,
    system_prompt=load_prompt("email-extractor-system-prompt").format(),
    output_type=EmailExtractor,
)

c:\Users\Lilit\Documents\GitHub\LLMOps\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
result = await email_extractor_agent.run(sample_mail)
result.output

EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_category='technical', urgency='high', summary='Oscar Johansson has been locked out of his account for three days due to invalid credentials errors. He has attempted multiple password reset requests but the reset emails are not arriving in his inbox, spam, or junk folders. This is preventing him from accessing important work documents needed to meet an upcoming Friday deadline. He has tried various troubleshooting steps including browser changes and cache clearing without success, and requests urgent assistance to regain access.')

## LLM Judge
1. require data with columns: inputs, expectations, outputs
2. mlflow experiments

In [10]:
result.output.model_dump()

{'sender_name': 'Oscar Johansson',
 'sender_email': 'oscar.johansson@yahoo.se',
 'issue_category': 'technical',
 'urgency': 'high',
 'summary': 'Oscar Johansson has been locked out of his account for three days due to invalid credentials errors. He has attempted multiple password reset requests but the reset emails are not arriving in his inbox, spam, or junk folders. This is preventing him from accessing important work documents needed to meet an upcoming Friday deadline. He has tried various troubleshooting steps including browser changes and cache clearing without success, and requests urgent assistance to regain access.'}

TODO: dataframe with inputs, expectations, outputs but only 1 row

In [11]:
df["outputs"] = [{},{},result.output.model_dump(), {}]
df

,inputs,expectations,outputs
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'excepted_response': '{""sender_name"": ""Erik L...",{}
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'excepted_response': '{""sender_name"": ""Maja B...",{}
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'excepted_response': '{""sender_name"": ""Oscar ...","{'sender_name': 'Oscar Johansson', 'sender_ema..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'excepted_response': '{""sender_name"": ""Linnea...",{}


In [12]:
df_sample = df.drop([0,1,3])
df_sample

,inputs,expectations,outputs
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'excepted_response': '{""sender_name"": ""Oscar ...","{'sender_name': 'Oscar Johansson', 'sender_ema..."
